# Classification de Sentiment en Fongbé
## Modèles de Machine Learning Classiques (TF-IDF, SMOTE, Naive Bayes, Random Forest, Soft Voting)

Ce notebook présente le pipeline complet d'entraînement et d'évaluation de modèles de machine learning pour la classification de sentiment sur un corpus de phrases en fongbé.

### Pipeline de traitement :
1. **Vectorisation TF-IDF de n-grams de caractères** (`char_wb`, (3, 5)) adaptée au Fongbé.
2. **Rééquilibre par SMOTE** (génération de données synthétiques) sur le jeu d'entraînement.
3. **Optimisation des hyperparamètres** (GridSearchCV pour Naive Bayes, Optuna pour Random Forest).
4. **Ajustement des seuils de décision (Threshold Tuning)** sur les probabilités du Soft Voting.
5. **Ré-entraînement final (85% Train+Val)** et évaluation ultime sur le jeu de test indépendant (15%).
6. **Exportation séparée** du vectorizer et des 3 modèles sous format `.joblib`.

| Label | Sentiment |
|-------|-----------|
| 0 | Neutre |
| 1 | Négatif |
| 2 | Positif |

## 1. Chargement des bibliothèques

In [2]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

from imblearn.over_sampling import SMOTE
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("Bibliothèques chargées avec succès.")

KeyboardInterrupt: 

## 2. Chargement et préparation du dataset

In [ ]:
df = pd.read_csv("../data/corpus_final.csv", sep="|")

df["sentiment_final"] = pd.to_numeric(df["sentiment_final"], errors="coerce")
df = df.dropna(subset=["fon", "sentiment_final"])
df["label"] = df["sentiment_final"].astype(int)

print(f"Nombre de phrases valides : {len(df):,}")
print("\nDistribution des classes :")
print(df["label"].value_counts().sort_index())
print("\nProportions (%) :")
print((df["label"].value_counts(normalize=True).sort_index() * 100).round(1))

LABEL_NAMES = {0: "Neutre", 1: "Négatif", 2: "Positif"}
LABEL_COLORS = {0: "#78909C", 1: "#EF5350", 2: "#66BB6A"}

plt.figure(figsize=(7, 4))
counts = df["label"].value_counts().sort_index()
bars = plt.bar(
    [LABEL_NAMES[i] for i in counts.index],
    counts.values,
    color=[LABEL_COLORS[i] for i in counts.index],
    edgecolor="white",
    linewidth=1.5,
)
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
             f"{val:,}", ha="center", va="bottom", fontweight="bold")

plt.title("Distribution des classes de sentiment", fontsize=14, fontweight="bold")
plt.ylabel("Nombre de phrases")
plt.tight_layout()
plt.show()

NameError: name 'pd' is not defined

## 3. Partitionnement des données (Train 70% / Validation 15% / Test 15%)

In [ ]:
X = np.array(df["fon"].astype(str).tolist(), dtype=object)
y = np.array(df["label"].tolist(), dtype=int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Jeu d'entraînement (Train) : {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Jeu de validation (Val)    : {len(X_val):,} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Jeu de test (Test)         : {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")
print(f"Total                      : {len(X):,}")

## 4. Vectorisation TF-IDF, SMOTE et Sélection de modèle

In [ ]:
# 1. Vectorisation TF-IDF par n-grams de caractères (3 à 5)
tfidf_select = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=40_000,
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = tfidf_select.fit_transform(X_train)
X_val_tfidf = tfidf_select.transform(X_val)

# 2. Application de SMOTE UNIQUEMENT sur le Train set
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_tfidf, y_train)

print(f"Distribution Train originale : {np.bincount(y_train)}")
print(f"Distribution Train après SMOTE : {np.bincount(y_train_res)}")

# 3. Multinomial Naive Bayes (GridSearchCV)
param_grid = {'alpha': [0.01, 0.05, 0.1, 0.5, 1.0]}
grid_mnb = GridSearchCV(MultinomialNB(), param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_mnb.fit(X_train_res, y_train_res)
best_alpha = grid_mnb.best_params_['alpha']
mnb = grid_mnb.best_estimator_
pred_val_mnb = mnb.predict(X_val_tfidf)

# 4. Random Forest (Optuna)
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 10, 60)
    rf_trial = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
    rf_trial.fit(X_train_res, y_train_res)
    preds = rf_trial.predict(X_val_tfidf)
    return accuracy_score(y_val, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)
best_rf_params = study.best_params

rf = RandomForestClassifier(**best_rf_params, random_state=42, n_jobs=-1)
rf.fit(X_train_res, y_train_res)
pred_val_rf = rf.predict(X_val_tfidf)

# 5. Soft Voting Classifier
voting_clf = VotingClassifier(
    estimators=[("mnb", mnb), ("rf", rf)],
    voting="soft",
    n_jobs=-1
)
voting_clf.fit(X_train_res, y_train_res)
pred_val_voting = voting_clf.predict(X_val_tfidf)

# 6. Bilan Validation
val_results = [
    {"Modèle": "Multinomial Naive Bayes", "Accuracy Val": accuracy_score(y_val, pred_val_mnb), "F1 Macro Val": f1_score(y_val, pred_val_mnb, average="macro")},
    {"Modèle": "Random Forest", "Accuracy Val": accuracy_score(y_val, pred_val_rf), "F1 Macro Val": f1_score(y_val, pred_val_rf, average="macro")},
    {"Modèle": "Soft Voting Classifier", "Accuracy Val": accuracy_score(y_val, pred_val_voting), "F1 Macro Val": f1_score(y_val, pred_val_voting, average="macro")},
]

df_val = pd.DataFrame(val_results)
print("\n" + "=" * 65)
print("ÉVALUATION SUR LE JEU DE VALIDATION")
print("=" * 65)
print(df_val.to_string(index=False))

best_architecture = df_val.sort_values(by="Accuracy Val", ascending=False).iloc[0]["Modèle"]
print(f"\nModèle sélectionné : {best_architecture}")

## 4.1 Ajustement des seuils de décision (Threshold Tuning) sur Validation

In [ ]:
# Récupération des probabilités brutes sur la validation
probs_val = voting_clf.predict_proba(X_val_tfidf)

# Poids personnalisés pour donner un léger bonus aux classes Positif et Négatif
CLASS_WEIGHTS = np.array([1.0, 1.30, 1.15])

probs_val_adjusted = probs_val * CLASS_WEIGHTS
preds_val_adjusted = np.argmax(probs_val_adjusted, axis=1)

print("Rapport de classification sur Validation (avec ajustement des seuils) :")
print(classification_report(y_val, preds_val_adjusted, target_names=["Neutre", "Négatif", "Positif"]))
print(f"Accuracy avec seuils ajustés : {accuracy_score(y_val, preds_val_adjusted):.4f}")

## 5. Ré-entraînement final sur Train + Validation (85%)

In [ ]:
X_train_val = np.concatenate([X_train, X_val])
y_train_val = np.concatenate([y_train, y_val])

# Vectorisation TF-IDF finale
tfidf_final = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=40_000,
    sublinear_tf=True,
    min_df=2
)
X_train_val_tfidf = tfidf_final.fit_transform(X_train_val)
X_test_tfidf = tfidf_final.transform(X_test)

# Application de SMOTE sur les 85% de données
smote_final = SMOTE(random_state=42)
X_train_val_res, y_train_val_res = smote_final.fit_resample(X_train_val_tfidf, y_train_val)

# Entraînement des 3 modèles finaux avec leurs hyperparamètres optimaux
mnb_m = MultinomialNB(alpha=best_alpha)
mnb_m.fit(X_train_val_res, y_train_val_res)

rf_m = RandomForestClassifier(**best_rf_params, random_state=42, n_jobs=-1)
rf_m.fit(X_train_val_res, y_train_val_res)

final_model = VotingClassifier(
    estimators=[("mnb", mnb_m), ("rf", rf_m)],
    voting="soft",
    n_jobs=-1
)
final_model.fit(X_train_val_res, y_train_val_res)

print(f"✅ Ré-entraînement final terminé sur {len(X_train_val):,} phrases.")

## 6. Évaluation sur le jeu de test

In [ ]:
# Prédiction avec probabilités et seuils ajustés sur le jeu de test
probs_test = final_model.predict_proba(X_test_tfidf)
probs_test_adjusted = probs_test * CLASS_WEIGHTS
y_test_pred = np.argmax(probs_test_adjusted, axis=1)

test_acc = accuracy_score(y_test, y_test_pred)
test_f1_macro = f1_score(y_test, y_test_pred, average="macro")
test_f1_weighted = f1_score(y_test, y_test_pred, average="weighted")

print("=" * 65)
print(f"RÉSULTATS DU JEU DE TEST — {best_architecture.upper()}")
print("=" * 65)
print(f"Accuracy       : {test_acc:.4f}")
print(f"F1-Score Macro  : {test_f1_macro:.4f}")
print(f"F1-Score Weighted: {test_f1_weighted:.4f}")
print()
print(classification_report(y_test, y_test_pred, target_names=["Neutre", "Négatif", "Positif"]))

cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Neutre", "Négatif", "Positif"],
            yticklabels=["Neutre", "Négatif", "Positif"])
plt.title(f"Matrice de confusion — {best_architecture}")
plt.xlabel("Prédiction")
plt.ylabel("Classe réelle")
plt.tight_layout()
plt.show()

## 7. Sauvegarde du vectorizer et des modèles

In [ ]:
os.makedirs("../models", exist_ok=True)

# Vectorizer
joblib.dump(tfidf_final, "../models/tfidf.joblib")

# Les 3 modèles entraînés sur 85% des données
joblib.dump(mnb_m, "../models/mnb_model.joblib")
joblib.dump(rf_m, "../models/rf_model.joblib")
joblib.dump(final_model, "../models/voting_model.joblib")

print("Fichiers sauvegardés dans models/ :")
print("   - tfidf.joblib          (vectorizer TF-IDF)")
print("   - mnb_model.joblib      (Multinomial Naive Bayes)")
print("   - rf_model.joblib       (Random Forest)")
print("   - voting_model.joblib   (Soft Voting Classifier)")